In [ ]:
# Cellule 1 - Charger les bibliothèques

import json
import shutil
from pathlib import Path

import joblib
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
# Cellule 2 - Charger les données

dataset = fetch_california_housing(as_frame=True)
df = dataset.frame

X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(df.shape)
df.head()

In [ ]:
# Cellule 3 - Fonction d’évaluation


def evaluate_pipeline(pipeline, X_test, y_test):
    preds = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    return {
        "mae": float(mae),
        "rmse": float(rmse),
        "r2": float(r2),
    }

In [ ]:
# Cellule 4 - Entraîner le modèle A (champion)

model_a = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            RandomForestRegressor(
                n_estimators=30,
                max_depth=8,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

model_a.fit(X_train, y_train)
metrics_a = evaluate_pipeline(model_a, X_test, y_test)
metrics_a

In [ ]:
# Cellule 5 - Entraîner le modèle B (challenger)

model_b = Pipeline(
    [
        ("scaler", StandardScaler()),
        (
            "model",
            GradientBoostingRegressor(
                n_estimators=80,
                learning_rate=0.05,
                random_state=42,
            ),
        ),
    ]
)

model_b.fit(X_train, y_train)
metrics_b = evaluate_pipeline(model_b, X_test, y_test)
metrics_b

In [ ]:
# Cellule 6 - Sauvegarder modèles et métriques

models_dir = Path("../artifacts/models")
metrics_dir = Path("../artifacts/metrics")
models_dir.mkdir(parents=True, exist_ok=True)
metrics_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(model_a, models_dir / "model_v1.joblib")
joblib.dump(model_b, models_dir / "model_v2.joblib")

with open(metrics_dir / "metrics_v1.json", "w", encoding="utf-8") as f:
    json.dump(metrics_a, f, indent=2)

with open(metrics_dir / "metrics_v2.json", "w", encoding="utf-8") as f:
    json.dump(metrics_b, f, indent=2)

shutil.copy(models_dir / "model_v1.joblib", models_dir / "model_champion.joblib")
shutil.copy(models_dir / "model_v2.joblib", models_dir / "model_challenger.joblib")

print("Modèles A et B sauvegardés")